In [2]:
# %%
from unsloth import FastLanguageModel
import torch
max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # 设置环境变量
# 基于unsloth加载Llama的蒸馏模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/root/autodl-tmp/DeepSeek-R1",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.16.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/root/autodl-tmp/DeepSeek-R1 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


In [3]:
# %%
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are an expert in reading comprehension and information extraction.
Given a context passage and a question, analyze the text carefully and provide the exact answer span from the passage.

### Question:
{}

### Response:
<think>{}"""

question = "Article: Phytochemistry is a branch of plant biochemistry primarily concerned with the chemical substances produced by plants during secondary metabolism. Some of these compounds are toxins such as the alkaloid coniine from hemlock. Others, such as the essential oils peppermint oil and lemon oil are useful for their aroma, as flavourings and spices (e.g., capsaicin), and in medicine as pharmaceuticals as in opium from opium poppies. Many medicinal and recreational drugs, such as tetrahydrocannabinol (active ingredient in cannabis), caffeine, morphine and nicotine come directly from plants. Others are simple derivatives of botanical natural products. For example, the pain killer aspirin is the acetyl ester of salicylic acid, originally isolated from the bark of willow trees, and a wide range of opiate painkillers like heroin are obtained by chemical modification of morphine obtained from the opium poppy. Popular stimulants come from plants, such as caffeine from coffee, tea and chocolate, and nicotine from tobacco. Most alcoholic beverages come from fermentation of carbohydrate-rich plant products such as barley (beer), rice (sake) and grapes (wine).Now answer this question: Where do some medicines and recreational drugs come from?"


FastLanguageModel.for_inference(model)
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=8192,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
Okay, so I need to figure out where some medicines and recreational drugs come from based on the provided passage. Let me read through the text carefully to understand the key points.

The passage starts by defining phytochemistry as a branch of plant biochemistry dealing with secondary metabolism in plants, producing compounds like toxins and useful substances. It mentions specific examples like coniine from hemlock, essential oils such as peppermint and lemon oil, capsaicin, and pharmaceuticals like opium. Then it goes on to discuss where medicines and drugs come from.

It says that many medicinal and recreational drugs come directly from plants. Examples include tetrahydrocannabinol from cannabis, caffeine from coffee, tea, and chocolate, nicotine from tobacco, morphine and opium. It also mentions derivatives, like aspirin being an acetyl ester of salicylic acid from willow trees, and heroin from chemical modification of morphine.

Additionally, it notes that popular stimul

In [10]:
# %%
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are an expert in reading comprehension and information extraction.
Given a context passage and a question, analyze the text carefully and provide the exact answer span from the passage.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

# 迭代训练集数据，处理prompt
def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

# from datasets import load_dataset
# dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'zh', split = "train[0:500]", trust_remote_code=True)
import json
from datasets import Dataset
# with open('/root/autodl-tmp/medical-o1-reasoning-SFT/medical_o1_sft_Chinese.json', 'r', encoding='utf-8') as f:
#     dataset = Dataset.from_list(list(json.load(f).values())[:500])
with open('/root/autodl-tmp/medical-o1-reasoning-SFT/medical_o1_sft_Chinese.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    # 假设 data 是一个列表
    dataset = Dataset.from_list(data[:500])
print(dataset.column_names)


['Question', 'Complex_CoT', 'Response']


In [11]:
# %%
dataset = dataset.map(formatting_prompts_func, batched = True)
print(dataset["text"][0])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are an expert in reading comprehension and information extraction.
Given a context passage and a question, analyze the text carefully and provide the exact answer span from the passage.

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造

In [13]:
# %%
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/root/autodl-tmp/DeepSeek-R1",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

# 训练
trainer_stats = trainer.train()

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.16.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/root/autodl-tmp/DeepSeek-R1 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.347900
2,2.227900
3,2.311000
4,2.434700
5,2.176600
6,2.035600
7,2.032500
8,1.867500
9,1.928500
10,1.813300


In [17]:
# %%
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
question = "一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？"

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
这个病人已经有急性阑尾炎的症状了，已经发病5天，腹痛虽有减轻，但仍然发热。这个情况有点奇怪，通常急性阑尾炎的疼痛应该会慢慢减轻，发热也应该会下降才对。可是现在看来，右下腹还有压痛的包块，这说明什么呢？

哦，可能是因为他的阑尾炎没好转，或者还有其他的并发症。比如说，阑尾炎有时候会有炎症性包块，或者是不是有其他的问题，比如说感染性炎症？这可能需要进一步的检查。

再想想，既然包块存在，且病人仍然发热，这可能提示有更严重的感染。或许需要用一些抗生素来治疗一下。对了，抗生素通常可以针对感染，帮助缓解发热和其他症状。

嗯，既然包块还存在，应该还要看看有没有其他的并发症。比如说，是否有感染扩散或者其他的炎症性问题。所以，可能需要结合体内的其他检查，比如说血液检查或者影像学检查。

哦，对了，发热和包块的存在，可能还会考虑到是否有脓液积聚或者其他的感染性问题。所以，可能需要一些影像学检查来了解具体情况。

嗯，综合来看，应该先确定是不是有感染扩散，或者其他的并发症存在。然后，根据具体情况来决定用什么样的抗生素和治疗方案。

哦，对了，包块和发热的存在，也可能提示需要考虑内脏炎或者其他的严重感染。所以，可能需要用一些针对性更强的检查，比如说超声或者CT来看看。

嗯，总之，先确定是不是有感染扩散或者其他的并发症存在，然后根据具体情况来决定用什么样的抗生素和治疗方案。

哦，还有，包块的存在可能需要进行外科检查，或者是考虑是否有其他的感染性问题需要处理。

总之，这个病人需要进一步的检查来明确情况，然后根据检查结果来制定治疗方案。

嗯，先进行一些检查，了解到底是什么情况，才能正确处理。
</think>
在这种情况下，患者已经发病5天，腹痛虽有减轻但仍然发热，并且右下腹有压痛的包块。这些症状提示可能存在感染性阑尾炎的并发症，例如炎症性包块、感染性炎症或其他感染性问题。

1. **评估并发症**：包块的存在可能意味着感染性阑尾炎的炎症性反应，或者是感染扩散的表现。为了更准确地确定病因，建议进行影像学检查，如超声或CT，以评估包块的具体位置和周围组织的状态。

2. **抗生素治疗**：鉴于患者仍然发热，可能需要使用抗生素来对抗感染。根据具体检查结果，选择适当的抗生素可能有助于缓解症状。

3. **外科检查**：如果包块是严重感染或脓液积聚的表现，可能需要进行

In [ ]:
# %%
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")